# 20 — Greedy MoE phase 2: expert architectures
Compare shared, full private-encoder, and rank-16 FiLM-adapter experts on seeds 0–2 using the same capacity and weight decay selected in phase 1.

In [ ]:
# ======================= EDIT THIS CELL ONLY =======================
GITHUB_OWNER='selimsidan'; GITHUB_REPO='dataset_moe_nids'; GITHUB_BRANCH='main'; GITHUB_SECRET_NAME='GITHUB_TOKEN'
DRIVE_DATA_DIR='/content/drive/MyDrive/NIDS_datasets'
DRIVE_OUTPUT_DIR='/content/drive/MyDrive/NIDS_analysis_outputs/dataset_moe_nids_runs'
STUDY_CONFIG='config/greedy_moe_study_v2.yaml'; PREFIX='nfv3_4way_greedy_v2'
PHASE='2'; EXECUTE=False; MAX_NEW_RUNS=None; RUN_TESTS=True
# ==================================================================

## Secure checkout and environment setup

In [ ]:
import base64, json, os, subprocess, sys
from pathlib import Path
from google.colab import drive, userdata
drive.mount('/content/drive'); token=userdata.get(GITHUB_SECRET_NAME)
if not token: raise RuntimeError(f'Add {GITHUB_SECRET_NAME} in Colab Secrets and grant access.')
auth=base64.b64encode(f'x-access-token:{token}'.encode()).decode(); git_env=os.environ|{'GIT_CONFIG_COUNT':'1','GIT_CONFIG_KEY_0':'http.https://github.com/.extraheader','GIT_CONFIG_VALUE_0':f'AUTHORIZATION: basic {auth}'}
repo=Path('/content')/GITHUB_REPO; url=f'https://github.com/{GITHUB_OWNER}/{GITHUB_REPO}.git'
subprocess.run(['git','-C',str(repo),'pull','--ff-only','origin',GITHUB_BRANCH] if (repo/'.git').is_dir() else ['git','clone','--branch',GITHUB_BRANCH,'--single-branch',url,str(repo)],env=git_env,check=True)
git_env.clear(); token=auth=None; os.chdir(repo)
os.environ.update(NIDS_DRIVE_BASE=DRIVE_DATA_DIR,NIDS_OUTPUT_DIR=DRIVE_OUTPUT_DIR,NIDS_SCRATCH_DIR='/content/dataset_moe_nids_scratch')
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-colab.txt'],check=True)
print('Commit:',subprocess.check_output(['git','rev-parse','--short','HEAD'],text=True).strip())

## Preflight, dry run, and optional execution

In [ ]:
import torch
from training.ablation_matrix import expected_evaluated_run_count, load_study
if not torch.cuda.is_available(): raise RuntimeError('Select a GPU runtime.')
study=load_study(STUDY_CONFIG); print('GPU:',torch.cuda.get_device_name(0),'| planned runs:',expected_evaluated_run_count(study))
if RUN_TESTS: subprocess.run([sys.executable,'-m','pytest','-q','tests/test_ablation_study.py','tests/test_private_encoder_moe.py','tests/test_active_dataset_toggle.py'],check=True)
command=[sys.executable,'-u','-m','training.ablation_matrix','--study-config',STUDY_CONFIG,'--phase',PHASE,'--prefix',PREFIX]; subprocess.run(command,check=True,env=os.environ.copy())
if EXECUTE:
    execute_command=[*command,'--execute']
    if MAX_NEW_RUNS is not None: execute_command += ['--max-new-runs',str(MAX_NEW_RUNS)]
    subprocess.run(execute_command,check=True,env=os.environ.copy())

## Review persisted state and summaries

In [ ]:
import pandas as pd
from IPython.display import display
summary_dir=Path(DRIVE_OUTPUT_DIR)/'results'/f'{PREFIX}_summary'; state_path=summary_dir/'study_state.json'
if state_path.is_file(): print(json.dumps(json.loads(state_path.read_text()).get('decisions',{}),indent=2))
for path in sorted(summary_dir.glob(f'Ablation_Phase{PHASE}_*.csv')): print(path.name); display(pd.read_csv(path))
print('Safe to rerun until phase status is complete:',summary_dir)

## Latent-space geometry, probes, and figures

In [ ]:
phase_runs_path = summary_dir / f'Ablation_Phase{PHASE}_Runs.csv'
metric_files = [
    'Latent_Snapshot_Metrics_By_Run.csv', 'Latent_Per_Class_By_Run.csv',
    'Latent_Probe_Metrics_By_Run.csv', 'Latent_Probe_Per_Class_By_Run.csv',
]
if phase_runs_path.is_file():
    from IPython.display import Image
    run_ids = set(pd.read_csv(phase_runs_path)['run_id'].astype(str))
    for name in metric_files:
        path = summary_dir / name
        if path.is_file():
            selected = pd.read_csv(path).query('run_id in @run_ids')
            print(name); display(selected)
    snapshot_path = summary_dir / 'Latent_Snapshot_Metrics_By_Run.csv'
    if snapshot_path.is_file():
        snapshots = pd.read_csv(snapshot_path).query('run_id in @run_ids')
        for figure_dir in sorted(snapshots['latent_figures_dir'].dropna().unique()):
            for png in sorted(Path(figure_dir).glob('*.png')):
                print(png.name); display(Image(filename=str(png)))
else:
    print('Phase run manifest is not available yet:', phase_runs_path)